# LAB | API Calling to ChatGPT - Automated Product Listing Generator

**Objective:** Generate e-commerce product listings from a product image and metadata using the OpenAI vision-capable `gpt-4o-mini` model.

**Instructions:** Run cells in order. Save the `.env` file **beside this notebook**, with `OPENAI_API_KEY=...`. Do **not** commit `.env`. The notebook generates `generated_listings.json` and the requested PNG evidence under `screenshots/` when live calls succeed.

**Important:** No API responses are pre-filled or invented. This notebook requires network access and valid OpenAI API billing/permissions. Placeholder prices are synthetic and are labelled accordingly.

## Step 0 — Install dependencies (once)
Install packages in the same Python environment as the notebook.

In [1]:
%pip install -q openai datasets pandas pillow python-dotenv httpx

Note: you may need to restart the kernel to use updated packages.


## Step 1 — Set up secure API access
Read `.env` using `python-dotenv`. Never print or commit the key. Confirm client creation (this does not by itself prove API authorization).

In [1]:
import os, json, base64, time, textwrap
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from openai import OpenAI, AuthenticationError, RateLimitError, APIConnectionError, APIStatusError

ROOT = Path.cwd()
SCREENSHOTS = ROOT / "screenshots"
SCREENSHOTS.mkdir(parents=True, exist_ok=True)
load_dotenv(ROOT / ".env")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY not found. Put it in an untracked .env file next to the notebook.")
client = OpenAI(api_key=api_key, timeout=45.0, max_retries=2)
MODEL = "gpt-4o-mini"  # Vision-capable model allowed by this course project
print("✓ OpenAI client created successfully (API request not yet tested).")

✓ OpenAI client created successfully (API request not yet tested).


### Screenshot helper
A small Python function draws **real program outputs** into readable PNG evidence. It does not fabricate successful calls or results.

In [2]:
def save_evidence_png(filename, heading, lines, max_width=106):
    """Save a legible image of actual text produced by notebook execution."""
    lines = [str(line) for line in lines]
    wrapped = []
    for line in lines:
        wrapped.extend(textwrap.wrap(line, width=max_width, replace_whitespace=False,
                                     drop_whitespace=False) or [""])
    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", 17)
        title_font = ImageFont.truetype("DejaVuSans.ttf", 25)
    except OSError:
        font = ImageFont.load_default()
        title_font = font
    width, line_height = 1500, 27
    height = max(200, min(3000, 115 + line_height * len(wrapped)))
    im = Image.new("RGB", (width, height), "#ffffff")
    draw = ImageDraw.Draw(im)
    draw.text((28, 18), heading, fill="#111827", font=title_font)
    draw.line((28, 62, width-28, 62), fill="#d1d5db", width=2)
    for i, line in enumerate(wrapped):
        y = 79 + i * line_height
        if y > height-24: break
        draw.text((28, y), line, fill="#374151", font=font)
    path = SCREENSHOTS / filename
    im.save(path)
    print(f"Evidence saved: {path}")
    return path

## Step 2 — Prepare the image dataset
Download 100 rows of `ashraq/fashion-product-images-small`. Each dataset row holds an actual PIL image, not an image path. Save the images to local JPEGs. Since the dataset has **no real prices**, use explicit synthetic category prices. If download fails, provide your own actual photos and a `local_products.json` with at least 3 entries (`id`, `name`, `category`, `price`, `image_path`).

In [3]:
from datasets import load_dataset

IMAGES_DIR = ROOT / "product_images"
IMAGES_DIR.mkdir(exist_ok=True)
CATEGORY_PRICES = {"Apparel": 29.99, "Accessories": 24.99,
                   "Footwear": 59.99, "Personal Care": 14.99}
try:
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")
    rows = []
    for n, row in enumerate(dataset):
        item_id = str(row.get("id", n))
        image_path = IMAGES_DIR / f"{item_id}.jpg"
        image = row.get("image")
        if image is None:
            continue
        image.convert("RGB").save(image_path, format="JPEG", quality=90)
        category = str(row.get("masterCategory") or row.get("category") or "Other")
        rows.append({"id": item_id,
                     "name": str(row.get("productDisplayName") or f"Product {item_id}"),
                     "category": category,
                     "price": CATEGORY_PRICES.get(category, 19.99),
                     "price_is_placeholder": True,
                     "additional_info": ", ".join(str(row[k]) for k in
                         ("subCategory", "articleType", "baseColour", "gender", "usage")
                         if row.get(k)),
                     "image_path": str(image_path)})
    products_df = pd.DataFrame(rows)
    data_source = "Hugging Face: ashraq/fashion-product-images-small"
except Exception as exc:
    print(f"Dataset unavailable ({type(exc).__name__}). Trying local_products.json.")
    local_file = ROOT / "local_products.json"
    if not local_file.exists():
        raise RuntimeError("Hugging Face download failed. Add local_products.json and matching images/ files, then rerun.") from exc
    products_df = pd.DataFrame(json.loads(local_file.read_text(encoding="utf-8")))
    products_df["price_is_placeholder"] = products_df.get("price_is_placeholder", False)
    products_df["additional_info"] = products_df.get("additional_info", "")
    data_source = "User-supplied local product images"

assert len(products_df) >= 3, "At least 3 actual product photos are required."
assert all(Path(p).is_file() for p in products_df["image_path"].head(5)), "Missing image file."
print(f"✓ Prepared {len(products_df)} products from {data_source}.")
display(products_df.head(5)[["id", "name", "category", "price", "image_path"]])

c:\Users\serge\miniconda3\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\serge\miniconda3\envs\bootcamp-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\serge\.cache\huggingface\hub\datasets--ashraq--fashion-product-images-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administ

✓ Prepared 100 products from Hugging Face: ashraq/fashion-product-images-small.


,id,name,category,price,image_path
0,15970,Turtle Check Men Navy Blue Shirt,Apparel,29.99,f:\UserData\Desktop\AI_Consulting_and_Integrat...
1,39386,Peter England Men Party Blue Jeans,Apparel,29.99,f:\UserData\Desktop\AI_Consulting_and_Integrat...
2,59263,Titan Women Silver Watch,Accessories,24.99,f:\UserData\Desktop\AI_Consulting_and_Integrat...
3,21379,Manchester United Men Solid Black Track Pants,Apparel,29.99,f:\UserData\Desktop\AI_Consulting_and_Integrat...
4,53759,Puma Men Grey T-shirt,Apparel,29.99,f:\UserData\Desktop\AI_Consulting_and_Integrat...


## Step 3 — Encode an image as Base64
Base64 transports actual JPEG bytes inside the API request. Never print an entire Base64 image into screenshots.

In [4]:
def encode_image_to_base64(image_path):
    """Read a JPEG image as bytes and return a Base64 ASCII string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)
print("✓ Encoded image:", Path(sample_path).name,
      "| Base64 prefix:", encoded_image[:24], "| characters:", len(encoded_image))

✓ Encoded image: 15970.jpg | Base64 prefix: /9j/4AAQSkZJRgABAQAAAQAB | characters: 3108


## Step 4 — Create the product listing prompt
Ask for JSON with title, description, 5–7 features and 10–15 SEO keywords. The model must not invent materials, certifications, origin, sizes, or technical specifications.

In [5]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    details = additional_info or "None supplied"
    return f"""You are an e-commerce copywriter. Examine the image and use only visible or supplied facts.
Product name: {product_name}
Price (USD; synthetic placeholder unless otherwise stated): ${float(price):.2f}
Category: {category}
Additional metadata: {details}
Return valid JSON ONLY with exactly these keys:
{{"title": "specific product title", "description": "2-3 factual sentences",
  "features": ["5 to 7 concise features"], "keywords": ["10 to 15 SEO keyword phrases"]}}
Use accurate visible colours/design elements. Do not invent materials, measurements,
performance claims, certifications, or brand details. Avoid unsupported promises.
The price is a demonstration placeholder, not a verified retail price.
"""

sample = products_df.iloc[0]
prompt = create_product_listing_prompt(sample["name"], sample["price"],
                                       sample["category"], sample.get("additional_info"))
print(prompt)

You are an e-commerce copywriter. Examine the image and use only visible or supplied facts.
Product name: Turtle Check Men Navy Blue Shirt
Price (USD; synthetic placeholder unless otherwise stated): $29.99
Category: Apparel
Additional metadata: Topwear, Shirts, Navy Blue, Men, Casual
Return valid JSON ONLY with exactly these keys:
{"title": "specific product title", "description": "2-3 factual sentences",
  "features": ["5 to 7 concise features"], "keywords": ["10 to 15 SEO keyword phrases"]}
Use accurate visible colours/design elements. Do not invent materials, measurements,
performance claims, certifications, or brand details. Avoid unsupported promises.
The price is a demonstration placeholder, not a verified retail price.



## Step 5 — Send a vision request and parse the response
Send one message containing two parts: text prompt and Base64 image URL. Parse JSON carefully, validate fields, and save a PNG **only after a successful live API response**.

In [6]:
def parse_json_response(content):
    """Accept strict JSON or JSON wrapped in Markdown fences."""
    cleaned = (content or "").strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else ""
        cleaned = cleaned.rsplit("```", 1)[0].strip()
    obj = json.loads(cleaned)
    if not isinstance(obj, dict):
        raise ValueError("Listing must be a JSON object")
    for field in ("title", "description", "features", "keywords"):
        if field not in obj:
            raise ValueError(f"Missing required field: {field}")
    if not isinstance(obj["features"], list) or not 5 <= len(obj["features"]) <= 7:
        raise ValueError("Expected 5–7 features")
    if not isinstance(obj["keywords"], list) or not 10 <= len(obj["keywords"]) <= 15:
        raise ValueError("Expected 10–15 keywords")
    return obj


def generate_product_listing(product, active_client=client):
    """Call the OpenAI Chat Completions API using a real product photo."""
    image_b64 = encode_image_to_base64(product["image_path"])
    prompt = create_product_listing_prompt(product["name"], product["price"],
                                           product["category"], product.get("additional_info"))
    response = active_client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {
                "url": f"data:image/jpeg;base64,{image_b64}", "detail": "low"}}
        ]}],
        temperature=0.3,
        response_format={"type": "json_object"},
        max_tokens=950,
    )
    listing = parse_json_response(response.choices[0].message.content)
    return {
        "product_id": str(product["id"]), "product_name": product["name"],
        "category": product["category"], "input_price_usd": float(product["price"]),
        "price_is_placeholder": bool(product.get("price_is_placeholder", True)),
        "image_path": product["image_path"], "model": MODEL,
        "api_response_id": response.id, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "listing": listing
    }

# LIVE request: this consumes API tokens. Do not replace output with mock data.
first_listing = generate_product_listing(products_df.iloc[0])
print("✓ Successful API call:", first_listing["api_response_id"])
print(json.dumps(first_listing["listing"], indent=2, ensure_ascii=False))
save_evidence_png("01_successful_api_call.png", "Step 5 — Successful LIVE OpenAI API call", [
    "Status: SUCCESS (actual API response)", "Model: " + MODEL,
    "Response ID: " + first_listing["api_response_id"],
    "Product: " + first_listing["product_name"],
    "Title: " + first_listing["listing"]["title"],
    "Features: " + str(len(first_listing["listing"]["features"])),
    "Keywords: " + str(len(first_listing["listing"]["keywords"]))])

✓ Successful API call: chatcmpl-EWp3BoWBNgS1o9JI4n7OPn51n3hVT
{
  "title": "Turtle Check Men Navy Blue Shirt",
  "description": "This casual navy blue shirt features a stylish check pattern, perfect for everyday wear. Designed for comfort and versatility, it can be easily paired with jeans or chinos.",
  "features": [
    "Navy blue color with check pattern",
    "Short sleeves for a relaxed fit",
    "Button-down collar for a polished look",
    "Casual style suitable for various occasions",
    "Easy to pair with different outfits",
    "Available in multiple sizes",
    "Machine washable for easy care"
  ],
  "keywords": [
    "men's navy blue shirt",
    "casual check shirt",
    "short sleeve shirt",
    "Turtle Check shirt",
    "men's apparel",
    "stylish men's shirt",
    "everyday wear shirt",
    "comfortable casual shirt",
    "button-down collar shirt",
    "fashionable men's topwear",
    "versatile men's clothing",
    "check pattern shirt",
    "men's summer shirt",
  

WindowsPath('f:/UserData/Desktop/AI_Consulting_and_Integration/Week_01/Day_4/lab-api-calling-chatgpt/screenshots/01_successful_api_call.png')

## Step 6 — Batch process five products
Continue when individual products fail. Save all successful listings atomically; create visual evidence of at least 3 real generated listings **only if 3 calls succeed**. Sleep between calls to reduce rate-limit pressure.

In [7]:
listings = [first_listing]
batch_errors = []
for index in range(1, min(5, len(products_df))):
    product = products_df.iloc[index]
    try:
        result = generate_product_listing(product)
        listings.append(result)
        print(f"✓ Product {index + 1}: {result['listing']['title']}")
    except (FileNotFoundError, AuthenticationError, RateLimitError,
            APIConnectionError, APIStatusError, ValueError, json.JSONDecodeError) as exc:
        batch_errors.append({"index": index, "error_type": type(exc).__name__,
                             "message": str(exc)[:180]})
        print(f"✗ Product {index + 1}: {type(exc).__name__} — proceeding to next product")
    finally:
        time.sleep(1)

output_file = ROOT / "generated_listings.json"
output_file.write_text(json.dumps(listings, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ Saved {len(listings)} actual generated listings to {output_file.name}")
if len(listings) < 3:
    print("WARNING: Fewer than 3 successful responses. Fix errors and rerun Step 6 before submitting.")
else:
    evidence_lines = []
    for i, item in enumerate(listings[:3], 1):
        detail = item["listing"]
        evidence_lines += [f"PRODUCT {i} / ID {item['product_id']}",
                           "Title: " + detail["title"],
                           "Description: " + detail["description"],
                           "Features: " + "; ".join(detail["features"]),
                           "Keywords: " + ", ".join(detail["keywords"]), ""]
    save_evidence_png("02_three_generated_listings.png",
                      "Step 6 — Three LIVE generated product listings", evidence_lines)
print("Batch errors:", json.dumps(batch_errors, indent=2))

✓ Product 2: Peter England Men Party Blue Jeans
✓ Product 3: Titan Women Silver Watch
✓ Product 4: Manchester United Men Solid Black Track Pants
✓ Product 5: Puma Men Grey T-shirt
✓ Saved 5 actual generated listings to generated_listings.json
Evidence saved: f:\UserData\Desktop\AI_Consulting_and_Integration\Week_01\Day_4\lab-api-calling-chatgpt\screenshots\02_three_generated_listings.png
Batch errors: []


## Step 7 — Error-handling demonstrations
Three intentionally triggered local examples: a missing image file, a **simulated** HTTP 401 authentication error (no real API request), and fenced JSON. The resulting PNG explicitly labels the authentication scenario as simulated.

In [8]:
import httpx
error_evidence = []

# Scenario A: missing file — real filesystem error.
try:
    encode_image_to_base64(ROOT / "images" / "definitely_missing.jpg")
except FileNotFoundError as exc:
    error_evidence.append("A. FileNotFoundError caught: " + type(exc).__name__)

# Scenario B: simulated invalid key 401 — safe, deterministic, no network/key leakage.
fake_request = httpx.Request("POST", "https://api.openai.com/v1/chat/completions")
fake_response = httpx.Response(401, request=fake_request,
                               json={"error": {"message": "Invalid API key"}})
try:
    raise AuthenticationError("Invalid API key (SIMULATED)", response=fake_response,
                              body=fake_response.json())
except AuthenticationError as exc:
    error_evidence.append("B. AuthenticationError caught: simulated HTTP 401 / invalid API key")

# Scenario C: fenced JSON fails without cleaning but succeeds with it.
test_listing = {"title": "Demo", "description": "Demo description",
                "features": ["a", "b", "c", "d", "e"],
                "keywords": [str(i) for i in range(10)]}
fenced_json = "```json\n" + json.dumps(test_listing) + "\n```"
try:
    json.loads(fenced_json)
except json.JSONDecodeError:
    error_evidence.append("C1. json.loads(fenced_json): JSONDecodeError caught")
cleaned = parse_json_response(fenced_json)
assert cleaned["title"] == "Demo"
error_evidence.append("C2. parse_json_response(fenced_json): success after fence removal")
for line in error_evidence:
    print(line)
save_evidence_png("03_error_handling.png", "Step 7 — Error handling (simulated auth test labelled)", error_evidence)

A. FileNotFoundError caught: FileNotFoundError
B. AuthenticationError caught: simulated HTTP 401 / invalid API key
C1. json.loads(fenced_json): JSONDecodeError caught
C2. parse_json_response(fenced_json): success after fence removal
Evidence saved: f:\UserData\Desktop\AI_Consulting_and_Integration\Week_01\Day_4\lab-api-calling-chatgpt\screenshots\03_error_handling.png


WindowsPath('f:/UserData/Desktop/AI_Consulting_and_Integration/Week_01/Day_4/lab-api-calling-chatgpt/screenshots/03_error_handling.png')

## Final verification & submission checklist

- Confirm `generated_listings.json` contains **at least 3 entries with real `api_response_id` values**.
- Confirm `screenshots/01_successful_api_call.png`, `02_three_generated_listings.png`, `03_error_handling.png` exist.
- Manually compare colours/design in generated text with original images; record findings in the report.
- Do **not** commit `.env`, API keys, downloaded cache, or notebook outputs containing confidential information.
- Submit a Pull Request to the assigned **`lab-api-calling-chatgpt` educational repository** (not `openai/openai-python`), and sync progress in the portal.
- The original lab requests a `.py` script. This submission provides both `.py` and `.ipynb` versions.